# 🧠 Agent-R v0 — GRPO RL Training (Structural Diversity Project)

**Goal:** Fine-tune SFT-trained Qwen2.5-3B-Instruct with GRPO (Group Relative Policy Optimization) using TRL.  
**Output:** `Agent-R-v0` checkpoint — the pre-co-evolution starting point.  

### Kaggle Setup:
- **GPU:** T4 x2 (select *GPU T4 x2* in Kaggle settings — gives ~30GB VRAM total, avoids OOM)
- **Persistence:** Keep session alive. Runtime ~5–7 hrs.
- **Datasets needed (add as Kaggle datasets before running):**
  - Your Qwen2.5-3B-Instruct model files
  - This SFT adapter zip (`SFT_ON_GSM8K_AND_MATH.zip`) or extracted folder

> ⚠️ **Why GRPO instead of PPO?**  
> PPO needs a separate critic/value model loaded simultaneously → OOM on T4.  
> GRPO computes relative rewards within a *group* of sampled outputs — no value model needed. Same RL signal, ~40% less VRAM.

## Cell 1 — Install Dependencies

In [ ]:
%%capture
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "trl==0.12.2",
    "peft==0.14.0",
    "transformers==4.48.3",
    "accelerate==1.3.0",
    "bitsandbytes==0.45.3",
    "datasets==3.2.0",
    "torch==2.5.1",
    "einops",
    "flash-attn==2.7.3",   # comment out if install fails — will fall back to SDPA
], check=False)

print("✅ Packages installed")

## Cell 2 — Imports & GPU Check

In [ ]:
import os, re, math, json, time, warnings
warnings.filterwarnings("ignore")

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, LoraConfig, get_peft_model
from datasets import load_dataset, Dataset
from trl import GRPOConfig, GRPOTrainer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {props.total_memory/1e9:.1f} GB")

## Cell 3 — Paths (edit if your dataset mount differs)

In [ ]:
import glob

# ── Base model ─────────────────────────────────────────────────────────────
# Adjust to wherever your Qwen dataset is mounted on Kaggle
BASE_MODEL_PATH = "/kaggle/input/qwen2-5-3b-instruct"   # <-- change if needed

# ── SFT LoRA adapter ───────────────────────────────────────────────────────
# If you added the zip as a dataset it will be extracted already
SFT_ADAPTER_PATH = "/kaggle/input/sft-on-gsm8k-and-math/agent_r_sft"  # <-- change if needed

# ── Output ─────────────────────────────────────────────────────────────────
OUTPUT_DIR = "/kaggle/working/agent_r_v0_grpo"
CHECKPOINT_NAME = "agent_r_v0"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Quick sanity checks
assert os.path.isdir(BASE_MODEL_PATH),  f"❌ Base model not found at {BASE_MODEL_PATH}"
assert os.path.isdir(SFT_ADAPTER_PATH), f"❌ SFT adapter not found at {SFT_ADAPTER_PATH}"
print(f"✅ Base model : {BASE_MODEL_PATH}")
print(f"✅ SFT adapter: {SFT_ADAPTER_PATH}")
print(f"✅ Output dir : {OUTPUT_DIR}")

## Cell 4 — Load Tokenizer + Model (4-bit QLoRA + SFT LoRA merged in)

In [ ]:
# ── Tokenizer ──────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    SFT_ADAPTER_PATH,        # tokenizer files are in the adapter folder too
    trust_remote_code=True,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # GRPO needs left-padding for generation

# ── 4-bit quantisation config ──────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# ── Load base model in 4-bit ───────────────────────────────────────────────
print("Loading base model (4-bit)…")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",   # safe fallback; swap to 'flash_attention_2' if flash-attn installed
)

# ── Attach SFT LoRA weights ────────────────────────────────────────────────
print("Loading SFT LoRA adapter…")
model = PeftModel.from_pretrained(base_model, SFT_ADAPTER_PATH, is_trainable=True)
model.print_trainable_parameters()
print("✅ Model ready")

## Cell 5 — Add a NEW LoRA adapter on top for GRPO training

This keeps the SFT weights frozen as the reference policy and trains a fresh LoRA on top — exactly what GRPO expects.

In [ ]:
from peft import get_peft_model

# Freeze SFT adapter; add fresh GRPO adapter
for p in model.parameters():
    p.requires_grad = False

grpo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model.add_adapter(grpo_lora_config, adapter_name="grpo")
model.set_adapter("grpo")
model.print_trainable_parameters()
print("✅ GRPO LoRA adapter added")

## Cell 6 — Dataset (GSM8K)

In [ ]:
raw = load_dataset("gsm8k", "main")
train_data = raw["train"]
held_out    = raw["test"].select(range(200))   # for early-stopping eval

SYSTEM_PROMPT = (
    "You are a careful mathematical reasoner. "
    "Think step by step. End your answer with: #### <number>"
)

def format_prompt(example):
    """Convert GSM8K row → chat-formatted prompt string."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": example["question"]},
    ]
    return {
        "prompt": tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        ),
        "answer": example["answer"],   # kept for reward function
    }

train_ds  = train_data.map(format_prompt, remove_columns=train_data.column_names)
held_ds   = held_out.map(format_prompt,   remove_columns=held_out.column_names)

print(f"Train: {len(train_ds)} | Held-out: {len(held_ds)}")
print("Sample prompt:\n", train_ds[0]["prompt"][:300])

## Cell 7 — Reward Functions

Two rewards combined:
1. **Format reward** — did the model produce a `####` final answer?
2. **Correctness reward** — does the extracted number match the ground truth?

In [ ]:
def extract_answer(text: str):
    """Pull out the number after #### (GSM8K convention)."""
    match = re.search(r"####\s*([\d,\.\-]+)", text)
    if match:
        return match.group(1).replace(",", "").strip()
    return None

def reward_format(completions, **kwargs):
    """Returns 0.2 if response contains ####, else 0.0."""
    rewards = []
    for comp in completions:
        text = comp[0]["content"] if isinstance(comp, list) else comp
        rewards.append(0.2 if "####" in text else 0.0)
    return rewards

def reward_correctness(completions, answer, **kwargs):
    """Returns 1.0 if extracted answer matches GT, else 0.0."""
    rewards = []
    for comp, gt in zip(completions, answer):
        text = comp[0]["content"] if isinstance(comp, list) else comp
        pred = extract_answer(text)
        gold = extract_answer(gt)
        rewards.append(1.0 if (pred is not None and pred == gold) else 0.0)
    return rewards

print("✅ Reward functions defined")

## Cell 8 — Early Stopping Helper

In [ ]:
from transformers import TrainerCallback

class EarlyStopOnPlateau(TrainerCallback):
    """Stops training if held-out accuracy doesn't improve for `patience` evals."""
    def __init__(self, patience=20, eval_every=10):
        self.patience   = patience
        self.eval_every = eval_every
        self.best       = -1.0
        self.no_improve = 0

    def _eval_accuracy(self, model, tokenizer, dataset, n=100):
        model.eval()
        correct = 0
        sample  = dataset.select(range(min(n, len(dataset))))
        for row in sample:
            inputs = tokenizer(row["prompt"], return_tensors="pt").to(model.device)
            with torch.no_grad():
                out = model.generate(
                    **inputs, max_new_tokens=256, do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )
            gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            if extract_answer(gen) == extract_answer(row["answer"]):
                correct += 1
        model.train()
        return correct / len(sample)

    def on_step_end(self, args, state, control, model=None, tokenizer=None, **kwargs):
        if state.global_step % self.eval_every != 0:
            return
        acc = self._eval_accuracy(model, tokenizer, held_ds)
        print(f"[Step {state.global_step}] Held-out accuracy: {acc:.3f} | best: {self.best:.3f}")
        if acc > self.best + 1e-4:
            self.best       = acc
            self.no_improve = 0
        else:
            self.no_improve += 1
            print(f"  No improvement for {self.no_improve} evals (patience={self.patience})")
            if self.no_improve >= self.patience:
                print("⏹  Early stopping triggered.")
                control.should_training_stop = True

print("✅ EarlyStop callback ready")

## Cell 9 — GRPO Config

In [ ]:
grpo_config = GRPOConfig(
    # ── Core GRPO ──────────────────────────────────────────────────────────
    num_generations=4,          # group size G: sample 4 completions per prompt
    max_prompt_length=512,
    max_completion_length=512,

    # ── Optimiser ──────────────────────────────────────────────────────────
    learning_rate=5e-6,
    per_device_train_batch_size=1,   # keep tiny — 4 completions × 1024 tokens
    gradient_accumulation_steps=4,   # effective batch = 4
    num_train_epochs=1,
    max_steps=200,                   # as specified in your plan
    warmup_steps=10,
    bf16=True,

    # ── KL penalty (keeps policy close to SFT reference) ───────────────────
    beta=0.04,

    # ── Logging / saving ───────────────────────────────────────────────────
    output_dir=OUTPUT_DIR,
    logging_steps=5,
    save_steps=50,
    save_total_limit=3,
    report_to="none",   # change to "wandb" if you want W&B logging

    # ── Memory ─────────────────────────────────────────────────────────────
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    dataloader_pin_memory=False,
)

print("✅ GRPOConfig ready")

## Cell 10 — Trainer Setup & Launch 🚀

In [ ]:
trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    reward_funcs=[reward_format, reward_correctness],
    callbacks=[EarlyStopOnPlateau(patience=20, eval_every=10)],
)

print("🚀 Starting GRPO training…")
t0 = time.time()
trainer.train()
print(f"\n✅ Training finished in {(time.time()-t0)/3600:.2f} hrs")

## Cell 11 — Save Agent-R-v0 Checkpoint

In [ ]:
final_path = os.path.join(OUTPUT_DIR, CHECKPOINT_NAME)
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)

# Also save a small metadata card
meta = {
    "checkpoint_name": CHECKPOINT_NAME,
    "base_model": BASE_MODEL_PATH,
    "sft_adapter": SFT_ADAPTER_PATH,
    "rl_method": "GRPO",
    "grpo_steps": trainer.state.global_step,
    "lora_r": 16,
    "lora_alpha": 32,
    "description": "Pre-co-evolution starting point for Structural Diversity project",
}
with open(os.path.join(final_path, "agent_r_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(f"✅ Agent-R-v0 saved to: {final_path}")
print(json.dumps(meta, indent=2))

## Cell 12 — Quick Sanity Eval

In [ ]:
model.eval()
test_questions = [
    "Janet's ducks lay 16 eggs per day. She eats 3 for breakfast every morning and bakes muffins for her friends every day with 4. How much does she make selling the rest at the farmers' market if she sells for $2/egg?",
    "A store sells apples for $1.20 each and oranges for $0.80 each. If Sarah buys 5 apples and 3 oranges, how much does she pay in total?",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": q},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=300, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Q: {q[:80]}…")
    print(f"A: {answer}\n{'─'*60}")

## 📋 Notes for Next Steps (Co-Evolution)

- **Agent-R-v0** is now your pre-co-evolution policy (Qwen2.5-3B + SFT LoRA + GRPO LoRA).
- Next: train structurally *diverse* agents (different LoRA targets, different reasoning prompts) and use cross-agent verification as a reward signal.
- Keep `agent_r_v0_grpo/agent_r_v0/` — it's the reference checkpoint.

### GPU Recommendation Summary
| Kaggle GPU | VRAM | Verdict |
|---|---|---|
| T4 x1 | 15 GB | ❌ OOM with 4-bit + GRPO groups |
| **T4 x2** | **30 GB** | ✅ Works — use this |
| P100 | 16 GB | ⚠️ Marginal, may OOM |
| A100 | 40 GB | ✅ Fastest but not always available |